# Realistic Multi-Step Grip Prediction (Autoregressive)

This notebook addresses Reviewer 1 Comment #1: the original multi-step model used oracle
future feature values (actual observed weather at T_{n+1}..T_{n+k}). In real deployment,
those future values are unknown. This notebook implements an autoregressive approach:

- Uses the same single-step RWIS-LSTM architecture (WINDOW=4)
- At inference time, future features are approximated via **persistence**:
  the last observed feature vector at T_n is repeated for all future steps
- Predicted grip values are fed back as input for subsequent steps

Results are compared against the original oracle approach to show the performance gap.

In [ ]:
import os
import time
import warnings
import logging

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import LeaveOneGroupOut, train_test_split, GridSearchCV
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error,
    mean_absolute_percentage_error, median_absolute_error,
    mean_squared_log_error, explained_variance_score,
    max_error, r2_score
)
from scikeras.wrappers import KerasRegressor
from openpyxl import Workbook, load_workbook

tf.get_logger().setLevel(logging.ERROR)
logging.getLogger('tensorflow').setLevel(logging.ERROR)

# --- BEGIN TAGS PATCH ---
from sklearn.base import BaseEstimator
if not hasattr(BaseEstimator, '__sklearn_tags__'):
    BaseEstimator.__sklearn_tags__ = lambda self: {}
if not hasattr(KerasRegressor, '__sklearn_tags__'):
    KerasRegressor.__sklearn_tags__ = lambda self: {}
# --- END TAGS PATCH ---

print('Imports OK')

In [ ]:
# ============================================================
# Global settings
# ============================================================
PREPROCESSED_PATH = '../preprocessed_rwis_data.xlsx'
WINDOW   = 4          # input sequence length (same as main model)
HORIZONS = [1, 2, 4, 8]  # prediction steps: 10, 20, 40, 80 minutes
SEEDS    = (42, 43, 44)
RESULTS_PATH = 'multistep_realistic_results.xlsx'

# Load preprocessed data
df_enc = pd.read_excel(PREPROCESSED_PATH, engine='openpyxl')
label_col    = 'SurfaceGrip'
feature_cols = [c for c in df_enc.columns
                if c not in ['SampleTime', 'RwisControllerIntId', 'time_diff', label_col]]
n_feat = len(feature_cols)

print(f'Data loaded: {len(df_enc)} rows, {n_feat} features')
print(f'Devices: {df_enc["RwisControllerIntId"].unique()}')

In [ ]:
# ============================================================
# Dataset builder  (single-step, same as main model)
# ============================================================
def build_rnn_dataset(df, feature_cols, label_col='SurfaceGrip', window=WINDOW):
    """
    Build sequences for the single-step RWIS-LSTM.
    Each sequence:
      Rows 0..window-1 : [features_t, grip_t]  (history)
      Row  window      : [features_{t+1}, grip_t]  (oracle: next-step features + last grip)
    Target: grip_{t+1}
    """
    X_seq, y_seq, groups = [], [], []
    for gid, grp in df.groupby('RwisControllerIntId'):
        grp = grp.reset_index(drop=True)
        for i in range(window, len(grp)):
            hist     = grp.iloc[i - window:i]
            tgt      = grp.iloc[i]
            feat     = hist[feature_cols].values          # (window, n_feat)
            hy       = hist[[label_col]].values           # (window, 1)
            seq_hist = np.hstack([feat, hy])              # (window, n_feat+1)
            curr_feat = tgt[feature_cols].values.reshape(1, -1)  # oracle T_{t+1} features
            curr_all  = np.hstack([curr_feat, hy[-1:]])   # (1, n_feat+1)
            seq       = np.vstack([seq_hist, curr_all])   # (window+1, n_feat+1)
            X_seq.append(seq)
            y_seq.append(tgt[label_col])
            groups.append(gid)
    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32), np.array(groups)


# ============================================================
# Data balancing
# ============================================================
def balance_y_distribution(X_seq, y, bins=50, max_per_bin=1000, random_state=42):
    np.random.seed(random_state)
    edges = np.linspace(np.min(y), np.max(y), bins + 1)
    idx   = np.digitize(y, bins=edges, right=True)
    sel   = []
    for b in range(1, bins + 1):
        ids = np.where(idx == b)[0]
        if len(ids) > max_per_bin:
            ids = np.random.choice(ids, max_per_bin, replace=False)
        sel.extend(ids)
    sel = np.array(sel)
    return X_seq[sel], y[sel]


# ============================================================
# Evaluation metrics
# ============================================================
def evaluate(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2}


# ============================================================
# RWIS-LSTM trainer (GridSearch over epochs only)
# ============================================================
def train_rwis_lstm(X_train, y_train):
    input_shape = X_train.shape[1:]

    def build_model():
        m = Sequential([
            LSTM(64, input_shape=input_shape),
            Dense(1)
        ])
        m.compile(optimizer='adam', loss='mse')
        return m

    reg = KerasRegressor(
        model=build_model, verbose=0, batch_size=32,
        callbacks=[EarlyStopping(monitor='loss', patience=20,
                                 restore_best_weights=True)]
    )
    grid = GridSearchCV(reg, {'epochs': [200, 300, 400]},
                        scoring='neg_mean_squared_error', cv=3, n_jobs=1)
    grid.fit(X_train, y_train)
    return grid.best_estimator_


print('Helper functions defined.')

In [ ]:
# ============================================================
# Oracle multi-step prediction (original approach)
# Builds a dataset where future features are included (oracle)
# ============================================================
def build_oracle_multistep_dataset(df, feature_cols, label_col, window, k):
    """
    Oracle: input uses actual observed features at T_{n+1}..T_{n+k}.
    Returns X (N, window+k, n_feat+1), y (N, k), groups (N,)
    """
    X_seq, y_seq, groups = [], [], []
    for gid, grp in df.groupby('RwisControllerIntId'):
        grp = grp.reset_index(drop=True)
        for i in range(window, len(grp) - k):
            hist = grp.iloc[i - window:i]
            feat = hist[feature_cols].values
            hy   = hist[[label_col]].values
            rows = [np.hstack([feat, hy])]  # history block

            prev_grip = hy[-1, 0]
            targets = []
            for step in range(1, k + 1):
                future = grp.iloc[i + step - 1]
                f_feat = future[feature_cols].values.reshape(1, -1)
                f_row  = np.hstack([f_feat, [[prev_grip]]])
                rows.append(f_row)
                prev_grip = grp.iloc[i + step - 1][label_col]  # oracle grip
                targets.append(grp.iloc[i + step - 1][label_col])

            X_seq.append(np.vstack(rows))
            y_seq.append(targets)
            groups.append(gid)
    return (np.array(X_seq, dtype=np.float32),
            np.array(y_seq, dtype=np.float32),
            np.array(groups))


# ============================================================
# Realistic autoregressive prediction
# Uses the TRAINED single-step model + persistence for features
# ============================================================
def predict_autoregressive(model, X_test_1step, k):
    """
    Autoregressive k-step prediction WITHOUT oracle features.

    Persistence assumption: future feature vectors = last observed feature vector (T_n).
    Grip is fed back from the previous prediction.

    Args:
        model      : trained single-step RWIS-LSTM
        X_test_1step : (N, window+1, n_feat+1)  -- the standard 1-step test sequences
                       Row -2: features at T_{n-1} (last true history row)
                       Row -1: oracle features at T_n + grip at T_{n-1}
        k          : number of future steps to predict

    Returns:
        preds : (N, k)  predicted grip values for steps 1..k
    """
    seq = X_test_1step.copy()

    # Replace oracle features in last row with persistence (second-to-last row features)
    # Row -1 features = T_n oracle  -->  Row -2 features = T_{n-1} (last observed)
    seq[:, -1, :-1] = seq[:, -2, :-1]
    # grip in last row stays as-is (it's already the last observed grip T_{n-1})

    all_preds = []
    for _ in range(k):
        y_pred = model.predict(seq, verbose=0).ravel()   # (N,)
        all_preds.append(y_pred)

        # Build next row: persistence features + predicted grip
        persist_feat = seq[:, -1, :-1]                   # (N, n_feat) -- unchanged
        new_grip     = y_pred.reshape(-1, 1)             # (N, 1)
        new_row      = np.hstack([persist_feat, new_grip])[:, np.newaxis, :]  # (N,1,n_feat+1)

        # Roll window: drop oldest step, append new row
        seq = np.concatenate([seq[:, 1:, :], new_row], axis=1)

    return np.column_stack(all_preds)  # (N, k)


print('Prediction functions defined.')

In [ ]:
# ============================================================
# Excel results helper
# ============================================================
def save_results(rows, path=RESULTS_PATH):
    """Append result rows to Excel file."""
    if not os.path.exists(path):
        wb = Workbook()
        ws = wb.active
        ws.title = 'Results'
        ws.append(['seed', 'fold', 'station', 'horizon_k',
                   'minutes_ahead', 'approach',
                   'MSE', 'RMSE', 'MAE', 'R2'])
        wb.save(path)

    wb = load_workbook(path)
    ws = wb['Results']
    for r in rows:
        ws.append(r)
    wb.save(path)


# ============================================================
# Build full dataset once (single-step sequences)
# ============================================================
print('Building dataset...')
X_all, y_all, groups_all = build_rnn_dataset(df_enc, feature_cols, label_col)
print(f'Dataset shape: X={X_all.shape}, y={y_all.shape}')
print(f'Input shape per sample: {X_all.shape[1:]}  (window+1={WINDOW+1} steps, n_feat+1={n_feat+1} cols)')

In [ ]:
# ============================================================
# Main LOGO loop
# For each seed x fold:
#   1. Train single-step RWIS-LSTM
#   2. Evaluate oracle multi-step (k=1,2,4,8)
#   3. Evaluate autoregressive multi-step (k=1,2,4,8)
# ============================================================

logo = LeaveOneGroupOut()
total_start = time.perf_counter()

for seed in SEEDS:
    print(f'\n========== SEED {seed} ==========')
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.autograph.set_verbosity(0)

    for fold, (tr_idx, te_idx) in enumerate(
            logo.split(X_all, y_all, groups_all), start=1):

        station = groups_all[te_idx][0]
        print(f'\n--- Fold {fold}: leave out station {station} ---')

        # ----- Prepare training data -----
        Xs_tr, y_tr = X_all[tr_idx], y_all[tr_idx]
        Xs_te, y_te = X_all[te_idx], y_all[te_idx]

        Xs_tr_bal, y_tr_bal = balance_y_distribution(
            Xs_tr, y_tr, random_state=seed)

        Xs_tr_sub, Xs_val, y_tr_sub, y_val = train_test_split(
            Xs_tr_bal, y_tr_bal, test_size=0.2, random_state=seed)

        # ----- Train single-step model -----
        print('  Training single-step RWIS-LSTM...')
        t0 = time.perf_counter()
        model = train_rwis_lstm(Xs_tr_sub, y_tr_sub)
        print(f'  Training done in {time.perf_counter()-t0:.0f}s')

        # Save model weights
        os.makedirs('models/RWIS_LSTM_realistic', exist_ok=True)
        model.model_.save(
            f'models/RWIS_LSTM_realistic/seed{seed}_fold{fold}.h5')

        rows_to_save = []

        for k in HORIZONS:
            minutes = k * 10

            # ---- Build oracle test data for this k ----
            # Use the device-specific test portion of df_enc
            test_device_df = df_enc[
                df_enc['RwisControllerIntId'] == station
            ].reset_index(drop=True)

            # Oracle sequences for horizon k
            Xo, yo, _ = build_oracle_multistep_dataset(
                test_device_df, feature_cols, label_col, WINDOW, k)

            # Predict with oracle model (feed oracle sequences step by step)
            # We use the last input row's oracle features for each step
            # and evaluate the final step prediction
            # Simplified: evaluate only the k-th step prediction
            # Input = Xo[:, :WINDOW+1, :] shifted to include future oracle features
            # For fairness: use the oracle sequence ending at step k
            # (use last window+1 rows of Xo)
            Xo_input = Xo[:, -WINDOW-1:, :]
            y_oracle_pred = model.predict(Xo_input, verbose=0).ravel()
            y_oracle_true = yo[:, -1]   # grip at step k

            met_oracle = evaluate(y_oracle_true, y_oracle_pred)
            rows_to_save.append([
                seed, fold, station, k, minutes, 'oracle',
                met_oracle['MSE'], met_oracle['RMSE'],
                met_oracle['MAE'],  met_oracle['R2']
            ])

            # ---- Autoregressive (realistic) prediction ----
            # Use the 1-step test sequences; predict k steps autoregressively
            # with persistence for future features
            # We need the ground-truth grip at step k for the same windows
            # Build a 1-step dataset on the test device to get matching y values
            X1_te, y1_te, _ = build_rnn_dataset(
                test_device_df, feature_cols, label_col)

            # Only evaluate windows where there are k more steps after
            # (trim last k windows to align with oracle)
            max_win = len(X1_te) - k
            if max_win <= 0:
                continue
            X1_trimmed = X1_te[:max_win]

            # Build ground-truth grip at step k for each window
            # y1_te[j] = grip at window j's target step
            # We need grip at (j + k - 1) for step k ahead
            y_true_k = np.array([
                test_device_df.iloc[WINDOW + j + k - 1][label_col]
                for j in range(max_win)
            ], dtype=np.float32)

            ar_preds = predict_autoregressive(model, X1_trimmed, k)
            y_ar_pred = ar_preds[:, -1]   # prediction at step k

            met_ar = evaluate(y_true_k, y_ar_pred)
            rows_to_save.append([
                seed, fold, station, k, minutes, 'autoregressive',
                met_ar['MSE'], met_ar['RMSE'],
                met_ar['MAE'],  met_ar['R2']
            ])

            print(f'  k={k} ({minutes}min) | '
                  f'Oracle   R2={met_oracle["R2"]:.3f} RMSE={met_oracle["RMSE"]:.4f} | '
                  f'AutoReg  R2={met_ar["R2"]:.3f} RMSE={met_ar["RMSE"]:.4f}')

        save_results(rows_to_save)

print(f'\nAll done. Total time: {time.perf_counter()-total_start:.0f}s')
print(f'Results saved to {RESULTS_PATH}')

In [ ]:
# ============================================================
# Summary table: mean R2 and RMSE across seeds and folds
# ============================================================
results_df = pd.read_excel(RESULTS_PATH)

summary = (
    results_df
    .groupby(['horizon_k', 'minutes_ahead', 'approach'])[['R2', 'RMSE', 'MSE']]
    .agg(['mean', 'std'])
    .round(4)
)
summary.columns = ['_'.join(c) for c in summary.columns]
print(summary.to_string())

# Save summary
with pd.ExcelWriter(RESULTS_PATH, engine='openpyxl',
                    mode='a', if_sheet_exists='replace') as writer:
    summary.to_excel(writer, sheet_name='Summary')
print('Summary saved.')

In [ ]:
# ============================================================
# Plot: Oracle vs Autoregressive R2 across horizons
# ============================================================
import matplotlib.pyplot as plt

plot_df = (
    results_df
    .groupby(['minutes_ahead', 'approach'])['R2']
    .agg(['mean', 'std'])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(7, 4.5))

colors = {'oracle': '#2980b9', 'autoregressive': '#e74c3c'}
labels = {'oracle': 'Oracle (observed future features)',
          'autoregressive': 'Realistic (autoregressive + persistence)'}
markers = {'oracle': 'o', 'autoregressive': 's'}

for approach, grp in plot_df.groupby('approach'):
    ax.errorbar(
        grp['minutes_ahead'], grp['mean'],
        yerr=grp['std'],
        label=labels[approach],
        color=colors[approach],
        marker=markers[approach],
        linewidth=2, markersize=7, capsize=4
    )

ax.set_xlabel('Prediction Horizon (minutes)', fontsize=13)
ax.set_ylabel('R²', fontsize=13)
ax.set_title('Multi-Step Grip Prediction: Oracle vs Realistic', fontsize=14)
ax.set_xticks([10, 20, 40, 80])
ax.legend(fontsize=11)
ax.grid(True, linestyle='--', alpha=0.5)
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.savefig('multistep_oracle_vs_realistic.png', dpi=150)
plt.show()
print('Plot saved.')